# Pandas — تنظيف ودمج وتلخيص البيانات + مشروع تحليل عملاء

يغطّي هذا النوتبوك:
- الجزء 1: اكتشاف ومعالجة القيم المفقودة (Titanic)
- الجزء 2: اكتشاف وحذف التكرارات
- الجزء 3: التلخيص بـ GroupBy (Titanic)
- الجزء 4: دمج الجداول (merge / concat)
- الجزء 5: المشروع التكاملي — تحليل عملاء متجر

كل الخلايا منفّذة والنتائج ظاهرة تحتها.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
titanic = sns.load_dataset("titanic")
print("titanic shape:", titanic.shape)

titanic shape: (891, 15)


## الجزء 1 — اكتشاف ومعالجة القيم المفقودة

### 1) عدد القيم المفقودة بكل عمود (من الأكثر للأقل)

In [2]:
missing = titanic.isnull().sum().sort_values(ascending=False)
print(missing)

deck           688
age            177
embarked         2
embark_town      2
sex              0
pclass           0
survived         0
fare             0
parch            0
sibsp            0
class            0
adult_male       0
who              0
alive            0
alone            0
dtype: int64


### 2) تعويض قيم العمر المفقودة بالمتوسط في عمود جديد `age_filled`

In [3]:
mean_age = titanic["age"].mean()
titanic["age_filled"] = titanic["age"].fillna(mean_age)
print(f"متوسط العمر المستخدم للتعويض: {mean_age:.2f}")
print("مفقودات age الأصلي:", titanic["age"].isnull().sum())
print("مفقودات age_filled:", titanic["age_filled"].isnull().sum())
titanic[["age", "age_filled"]].head(7)

متوسط العمر المستخدم للتعويض: 29.70
مفقودات age الأصلي: 177
مفقودات age_filled: 0


,age,age_filled
0,22.0,22.000000
1,38.0,38.000000
2,26.0,26.000000
3,35.0,35.000000
4,35.0,35.000000
5,NaN,29.699118
6,54.0,54.000000


### 3) عمود `deck`: حذف أم إبقاء؟

**القرار: نحذف العمود.** عدد القيم المفقودة في `deck` هو 688 من أصل 891 (حوالي
77%)، وهذه نسبة عالية جدًا بحيث أن أي تعويض سيكون تخمينًا يشوّه البيانات؛ لذلك
الأنسب حذف العمود كاملًا لأنه لا يحمل معلومات كافية يُعتمد عليها.

In [4]:
print("نسبة المفقود في deck: %.1f%%" % (titanic["deck"].isnull().mean()*100))
titanic_no_deck = titanic.drop(columns=["deck"])
print("الأعمدة قبل:", titanic.shape[1], "| بعد حذف deck:", titanic_no_deck.shape[1])

نسبة المفقود في deck: 77.2%
الأعمدة قبل: 16 | بعد حذف deck: 15


### 4) تعويض `embarked` بالقيمة الأكثر تكرارًا (Mode) لأنه عمود نصي

In [5]:
mode_embarked = titanic["embarked"].mode()[0]
print("القيمة الأكثر تكرارًا في embarked:", mode_embarked)
titanic["embarked_filled"] = titanic["embarked"].fillna(mode_embarked)
print("مفقودات embarked الأصلي:", titanic["embarked"].isnull().sum())
print("مفقودات embarked_filled:", titanic["embarked_filled"].isnull().sum())

القيمة الأكثر تكرارًا في embarked: S
مفقودات embarked الأصلي: 2
مفقودات embarked_filled: 0


## الجزء 2 — اكتشاف وحذف التكرارات

### 1) DataFrame لـ6 عملاء مع تكرار صفين عمدًا

In [6]:
clients = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6],
    "name": ["أحمد", "سارة", "محمد", "ليان", "خالد", "نور"],
    "city": ["عمان", "إربد", "الزرقاء", "عمان", "إربد", "عمان"],
})
# تكرار صفين بالضبط (نسخ متعمّد)
clients = pd.concat([clients, clients.iloc[[1, 3]]], ignore_index=True)
print("عدد الصفوف مع التكرار:", clients.shape[0])
clients

عدد الصفوف مع التكرار: 8


,customer_id,name,city
0,1,أحمد,عمان
1,2,سارة,إربد
2,3,محمد,الزرقاء
3,4,ليان,عمان
4,5,خالد,إربد
5,6,نور,عمان
6,2,سارة,إربد
7,4,ليان,عمان


### 2) اكتشاف الصفوف المكررة قبل الحذف

In [7]:
dup_mask = clients.duplicated()
print("عدد الصفوف المكررة:", dup_mask.sum())
print("الصفوف المكررة:")
clients[dup_mask]

عدد الصفوف المكررة: 2
الصفوف المكررة:


,customer_id,name,city
6,2,سارة,إربد
7,4,ليان,عمان


### 3) حذف التكرارات مع الاحتفاظ بأول ظهور

In [8]:
clients_clean = clients.drop_duplicates(keep="first")
print("قبل:", clients.shape[0], "-> بعد:", clients_clean.shape[0])
clients_clean

قبل: 8 -> بعد: 6


,customer_id,name,city
0,1,أحمد,عمان
1,2,سارة,إربد
2,3,محمد,الزرقاء
3,4,ليان,عمان
4,5,خالد,إربد
5,6,نور,عمان


## الجزء 3 — تلخيص البيانات بـ GroupBy (Titanic)

### 1) متوسط سعر التذكرة لكل فئة ركاب

In [9]:
fare_by_class = titanic.groupby("pclass")["fare"].mean()
print(fare_by_class)
print("\nالفئة الأولى دفعت الأكثر بالمتوسط (%.2f)." % fare_by_class.loc[1])

pclass
1    84.154687
2    20.662183
3    13.675550
Name: fare, dtype: float64

الفئة الأولى دفعت الأكثر بالمتوسط (84.15).


### 2) نسبة النجاة لكل جنس

In [10]:
print(titanic.groupby("sex")["survived"].mean())

sex
female    0.742038
male      0.188908
Name: survived, dtype: float64


### 3) GroupBy مزدوج: نسبة النجاة حسب الجنس والفئة

In [11]:
by_sex_class = titanic.groupby(["sex", "pclass"])["survived"].mean()
print(by_sex_class)
print("\nأعلى نسبة نجاة:", by_sex_class.idxmax(), "= %.3f" % by_sex_class.max())
print("أدنى نسبة نجاة:", by_sex_class.idxmin(), "= %.3f" % by_sex_class.min())

sex     pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
Name: survived, dtype: float64

أعلى نسبة نجاة: ('female', np.int64(1)) = 0.968
أدنى نسبة نجاة: ('male', np.int64(3)) = 0.135


**الإجابة:** أعلى نسبة نجاة كانت لـ **الإناث في الفئة الأولى (96.8%)**، وأدنى
نسبة كانت لـ **الذكور في الفئة الثالثة (13.5%)**.

### 4) عدد الركاب حسب ميناء الصعود

In [12]:
print(titanic["embarked"].value_counts())

embarked
S    644
C    168
Q     77
Name: count, dtype: int64


## الجزء 4 — دمج الجداول: Merge / Concat

In [13]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name": ["أحمد", "سارة", "محمد", "ليان"],
    "city": ["عمان", "إربد", "الزرقاء", "عمان"],
})
orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 2, 1, 3, 5],   # العميل 5 غير موجود في جدول العملاء
    "amount": [150, 200, 90, 300, 60],
})
print("customers:"); print(customers)
print("\norders:"); print(orders)

customers:
   customer_id  name     city
0            1  أحمد     عمان
1            2  سارة     إربد
2            3  محمد  الزرقاء
3            4  ليان     عمان

orders:
   order_id  customer_id  amount
0       101            1     150
1       102            2     200
2       103            1      90
3       104            3     300
4       105            5      60


### 1) inner merge

In [14]:
inner = pd.merge(customers, orders, on="customer_id", how="inner")
inner

,customer_id,name,city,order_id,amount
0,1,أحمد,عمان,101,150
1,1,أحمد,عمان,103,90
2,2,سارة,إربد,102,200
3,3,محمد,الزرقاء,104,300


**ماذا حصل للطلب 105 (العميل 5)؟** اختفى، لأن `inner` يبقي فقط الصفوف التي لها
`customer_id` موجود في **الجدولين**، والعميل 5 غير موجود في جدول العملاء.

### 2) left merge (العملاء هم الأساس)

In [15]:
left = pd.merge(customers, orders, on="customer_id", how="left")
left

,customer_id,name,city,order_id,amount
0,1,أحمد,عمان,101.0,150.0
1,1,أحمد,عمان,103.0,90.0
2,2,سارة,إربد,102.0,200.0
3,3,محمد,الزرقاء,104.0,300.0
4,4,ليان,عمان,NaN,NaN


**الفرق عن inner:** كل العملاء يظهرون حتى لو بلا طلبات (مثل العميل 4 ليان تظهر
قيمه فارغة NaN)، لكن الطلب 105 للعميل 5 ما زال غير ظاهر لأنه ليس ضمن العملاء.

### 3) outer merge

In [16]:
outer = pd.merge(customers, orders, on="customer_id", how="outer")
outer

,customer_id,name,city,order_id,amount
0,1,أحمد,عمان,101.0,150.0
1,1,أحمد,عمان,103.0,90.0
2,2,سارة,إربد,102.0,200.0
3,3,محمد,الزرقاء,104.0,300.0
4,4,ليان,عمان,NaN,NaN
5,5,NaN,NaN,105.0,60.0


**الجديد في outer:** ظهر الطلب 105 للعميل 5 (مع اسم/مدينة فارغة)، إضافةً إلى
العميل 4 بلا طلبات. **الخلاصة:** `inner` = التقاطع فقط (المشترك بين الجدولين)،
`left` = كل العملاء + طلباتهم إن وُجدت، `outer` = كل شيء من الجدولين معًا مع
تعبئة الفراغات بـ NaN.

### 4) concat لدمج طلبات فبراير تحت الطلبات الأصلية

In [17]:
orders_feb = pd.DataFrame({
    "order_id": [106, 107, 108],
    "customer_id": [2, 3, 1],
    "amount": [120, 75, 210],
})
all_orders = pd.concat([orders, orders_feb], ignore_index=True)
print("عدد الطلبات بعد الدمج:", all_orders.shape[0])
all_orders

عدد الطلبات بعد الدمج: 8


,order_id,customer_id,amount
0,101,1,150
1,102,2,200
2,103,1,90
3,104,3,300
4,105,5,60
5,106,2,120
6,107,3,75
7,108,1,210


## الجزء 5 — المشروع التكاملي: تحليل عملاء متجر

### 1) جدول 8 عملاء مع قيم مفقودة في city وصف مكرر

In [18]:
customers5 = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "name": ["أحمد", "سارة", "محمد", "ليان", "خالد", "نور", "عمر", "هدى"],
    "city": ["عمان", "إربد", "الزرقاء", "عمان", None, "عمان", None, "إربد"],
})
# صف مكرر بالكامل للعميل 3 (محمد)
customers5 = pd.concat([customers5, customers5.iloc[[2]]], ignore_index=True)
print("الشكل مع التكرار والمفقودات:", customers5.shape)
print("مفقودات city:", customers5["city"].isnull().sum(), "| صفوف مكررة:", customers5.duplicated().sum())
customers5

الشكل مع التكرار والمفقودات: (9, 3)
مفقودات city: 2 | صفوف مكررة: 1


,customer_id,name,city
0,1,أحمد,عمان
1,2,سارة,إربد
2,3,محمد,الزرقاء
3,4,ليان,عمان
4,5,خالد,NaN
5,6,نور,عمان
6,7,عمر,NaN
7,8,هدى,إربد
8,3,محمد,الزرقاء


### 2) جدول 15 طلبًا موزّعة على العملاء (بعضهم بلا طلبات)

In [19]:
orders5 = pd.DataFrame({
    "order_id": list(range(101, 116)),
    "customer_id": [1, 1, 1, 2, 2, 3, 4, 4, 4, 4, 5, 5, 7, 7, 1],
    "amount": [150, 200, 90, 300, 60, 120, 500, 50, 75, 25, 80, 40, 220, 180, 60],
})
print("عدد الطلبات:", orders5.shape[0])
orders5.head()

عدد الطلبات: 15


,order_id,customer_id,amount
0,101,1,150
1,102,1,200
2,103,1,90
3,104,2,300
4,105,2,60


### 3) تنظيف الجدول: تعبئة city بـ "غير محدد" وحذف التكرارات

In [20]:
customers5["city"] = customers5["city"].fillna("غير محدد")
customers5 = customers5.drop_duplicates()
print("بعد التنظيف:", customers5.shape, "| مفقودات city:", customers5["city"].isnull().sum(),
      "| مكررات:", customers5.duplicated().sum())
customers5

بعد التنظيف:

 (8, 3) | مفقودات city: 0 | مكررات: 0


,customer_id,name,city
0,1,أحمد,عمان
1,2,سارة,إربد
2,3,محمد,الزرقاء
3,4,ليان,عمان
4,5,خالد,غير محدد
5,6,نور,عمان
6,7,عمر,غير محدد
7,8,هدى,إربد


### 4) دمج left (كل العملاء يظهرون حتى بلا طلبات)

In [21]:
merged5 = customers5.merge(orders5, on="customer_id", how="left")
print("شكل الجدول المدموج:", merged5.shape)
merged5.head(20)

شكل الجدول المدموج: (17, 5)


,customer_id,name,city,order_id,amount
0,1,أحمد,عمان,101.0,150.0
1,1,أحمد,عمان,102.0,200.0
2,1,أحمد,عمان,103.0,90.0
3,1,أحمد,عمان,115.0,60.0
4,2,سارة,إربد,104.0,300.0
5,2,سارة,إربد,105.0,60.0
6,3,محمد,الزرقاء,106.0,120.0
7,4,ليان,عمان,107.0,500.0
8,4,ليان,عمان,108.0,50.0
9,4,ليان,عمان,109.0,75.0


### 5) لكل عميل: عدد الطلبات ومجموع المبالغ

In [22]:
summary = merged5.groupby(["customer_id", "name"]).agg(
    orders_count=("order_id", "count"),
    total_spent=("amount", "sum"),
).reset_index()
summary

,customer_id,name,orders_count,total_spent
0,1,أحمد,4,500.0
1,2,سارة,2,360.0
2,3,محمد,1,120.0
3,4,ليان,4,650.0
4,5,خالد,2,120.0
5,6,نور,0,0.0
6,7,عمر,2,400.0
7,8,هدى,0,0.0


### 6) أفضل 3 عملاء حسب إجمالي الصرف

In [23]:
top_customers = summary.sort_values("total_spent", ascending=False).head(3)
top_customers[["name", "orders_count", "total_spent"]]

,name,orders_count,total_spent
3,ليان,4,650.0
0,أحمد,4,500.0
6,عمر,2,400.0


### 7) الخلاصة

أفضل 3 عملاء حسب إجمالي الصرف هم: **ليان (650)** ثم **أحمد (500)** ثم **عمر
(400)**. أبرز ملاحظة أن العميلين **نور** و**هدى** لم يقدّما أي طلب (عدد طلبات = 0
ومجموع صرف = 0)، وقد ظهرا في التحليل فقط لأننا استخدمنا `left merge`؛ وهذا مهم
تجاريًا لأنهما عملاء غير نشطين يمكن استهدافهما بعروض لتحفيزهما على الشراء. كما
نلاحظ أن العميل الأعلى صرفًا (ليان) حقّق ذلك من 4 طلبات، أي أن كثرة الطلبات
مرتبطة بارتفاع إجمالي الصرف.